# Visualize the shared regularizer transformation strength

Choose a dataset and `TRANSFORMATION_STRENGTH` below. The notebook selects fixed source anchors as `Experiment.adapt()` does, and calibrates the configured transformations using those anchors. Images are columns and transformations are rows. The final cell recomputes every achieved RMS displacement $s_k$ over all anchors.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import torch


def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
SOURCE_DIR = REPO_ROOT / "src"
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from domain_knowledge_analysis import utils
from domain_knowledge_analysis.utils.dataset_utils import create_dataset_transformation

print(f"Repository: {REPO_ROOT}")

## Settings

Set `DATASET` to `"mnist"`, `"lines_dataset"` (horizontal), `"vertical_lines_dataset"` (vertical), or `"ellipsis_dataset"` (ellipses), then run all cells. The shared strength $s$ is the fraction of each transformation's maximum anchor RMS: values near 0 are mild and $s=1$ is the earliest strongest valid modification. Use a value in (0, 1].

All choices use the transformation list, directions, and anchor count from `config/lora_vae_mnist.yaml`; the lines and ellipsis choices use dataset settings from `config/vae_lines.yaml`. A transformation that cannot change the selected anchors is shown as unchanged and skipped during calibration. In particular, some pixel-intensity transformations do not change binary lines.


In [ ]:
DATASET = "mnist"  # "mnist", "lines_dataset", "vertical_lines_dataset", or "ellipsis_dataset"
TRANSFORMATION_STRENGTH = 0.5
NUMBER_OF_IMAGES = 5
DEVICE = torch.device("cpu")


In [ ]:
dataset_name = DATASET.lower()
if dataset_name not in {"mnist", "lines_dataset", "vertical_lines_dataset", "ellipsis_dataset"}:
    raise ValueError("DATASET must be 'mnist', 'lines_dataset', 'vertical_lines_dataset', or 'ellipsis_dataset'.")

regularizer_config = utils.load_config(REPO_ROOT / "config" / "lora_vae_mnist.yaml")
if dataset_name == "mnist":
    config = regularizer_config
else:
    config = utils.load_config(REPO_ROOT / "config" / "vae_lines.yaml")
    config["lora"] = regularizer_config["lora"]
config["dataset"]["name"] = dataset_name
config["lora"]["regularizer"].pop("target_input_rms", None)
config["lora"]["regularizer"]["transformation_strength"] = TRANSFORMATION_STRENGTH

anchor_points = utils.select_anchor_points(config, dataset_name, DEVICE)
regularizer = config["lora"]["regularizer"]
batch_size = regularizer.get("calibration_batch_size", 100)

# These transformations can be exact identities on binary images.
intensity_transformations = {"contrast", "gamma_correction", "binarization", "sharpen"}
active = []
skipped = []
for item in regularizer["transformations"]:
    if item["name"] in intensity_transformations:
        transformation = create_dataset_transformation(item["name"], item["intensity"])
        if utils.measure_input_rms(transformation, anchor_points, batch_size) == 0:
            skipped.append(item["name"])
            continue
    active.append(item)
regularizer["transformations"] = active

print(f"Dataset: {dataset_name}")
print(f"Anchors: {len(anchor_points)}")
print(f"Transformation strength: {TRANSFORMATION_STRENGTH:g}")
if skipped:
    print(f"Skipped (no change to anchors): {', '.join(skipped)}")

transformations, _ = utils.create_regularizer_transformations(config, DEVICE, anchor_points)
calibration = regularizer["calibration"]
display_indices = torch.linspace(0, len(anchor_points) - 1, NUMBER_OF_IMAGES).long()
display_images = anchor_points[display_indices]


In [ ]:
def show_image(axis, image):
    image = image.detach().cpu()
    axis.imshow(image[0].numpy(), cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    axis.set_xticks([])
    axis.set_yticks([])


rows = len(transformations) + len(skipped) + 1
figure, axes = plt.subplots(rows, NUMBER_OF_IMAGES, figsize=(2.1 * NUMBER_OF_IMAGES, 1.8 * rows), squeeze=False)

for column, image in enumerate(display_images):
    show_image(axes[0, column], image)
    axes[0, column].set_title(f"Image {column + 1}")
axes[0, 0].set_ylabel("Original", rotation=0, ha="right", va="center", labelpad=12)

for row, (transformation, metadata) in enumerate(zip(transformations, calibration["transformations"]), start=1):
    transformed_images = transformation(display_images)
    for column, image in enumerate(transformed_images):
        show_image(axes[row, column], image)
    axes[row, 0].set_ylabel(metadata["name"].replace("_", " "), rotation=0, ha="right", va="center", labelpad=12)

for row, name in enumerate(skipped, start=len(transformations) + 1):
    for column, image in enumerate(display_images):
        show_image(axes[row, column], image)
    axes[row, 0].set_ylabel(f"{name.replace('_', ' ')} (no effect)", rotation=0, ha="right", va="center", labelpad=12)

figure.suptitle(f"Regularizer transformations at relative strength s = {TRANSFORMATION_STRENGTH:g}", fontsize=14)
figure.tight_layout(rect=[0, 0, 1, 0.985])
plt.show()

## Achieved input RMS $s_k$

These values are recomputed from the transformed pixels over every fixed anchor, channel and pixel.

In [ ]:
batch_size = config["lora"]["regularizer"].get("calibration_batch_size", 100)
print(f"{'transformation':30} {'intensity':>12} {'max intensity':>14} {'max s_k':>12} {'computed s_k':>16} {'fraction':>12}")
print("-" * 104)

for transformation, metadata in zip(transformations, calibration["transformations"]):
    achieved_rms = utils.measure_input_rms(transformation, anchor_points, batch_size)
    relative_strength = achieved_rms / metadata["maximum_input_rms"]
    print(f"{metadata['name']:30} {metadata['resolved_intensity']:12.6g} {metadata['maximum_intensity']:14.6g} {metadata['maximum_input_rms']:12.6f} {achieved_rms:16.8f} {relative_strength:12.3f}")